# Checkup figures

Makes the figures for the Nov 4 checkup deck from the specs in [`docs/CHECKUP-VISUALS.md`](../docs/CHECKUP-VISUALS.md). Every figure is saved under `docs/figures/` with the cell that made it, so anyone can re-run it.

| Cell | Figure | Deck slot | Needs the data |
|---|---|---|---|
| A | per-class IoU bars, ours vs the paper | slide 7 | no |
| B | architecture strip, current vs goal | slide 8 (fallback for the Slides drawing) | no |
| C | one cube: false colour, mask, mean spectrum per class | slide 6 or 3 | yes |
| D | mean spectrum per class over N train cubes, with the 14 token groups | slide 3 or 6 | yes |
| G | training curves from `metrics.csv` | Nov 5 | a metrics file |

Numbers in A come from `docs/RESULTS.md` (row `cnn_ds2`) and Casao et al., IROS 2024, Table IV. Shapes in B come from `src/sw/models/transformer.py` on `main`. Cells C and D read the dataset through `SW_ROOT`, exactly like notebook 01, and skip themselves if it is not set.

Class colours are the dataset palette from `meta.json` (`sw.data.PALETTE`), so the charts agree with the mask. On a white surface film, basket, tape and bag have low contrast, so every bar carries an ink edge and a printed value, and every line carries a direct label.

## Colab setup

Skip on your own machine. On Colab it clones the repo and installs the libraries. The 23 GB download runs only when `DOWNLOAD_DATA` is True; cells A, B and G do not need it.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DOWNLOAD_DATA = False  # True only for cells C and D (about 23 GB)
if IN_COLAB:
    os.chdir("/content")
    repo = Path("/content/255-final-project")
    if not repo.is_dir():
        subprocess.check_call(
            ["git", "clone", "https://github.com/scottn66/255-final-project.git", str(repo)]
        )
    os.chdir(repo)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    if DOWNLOAD_DATA and not os.environ.get("SW_ROOT"):
        subprocess.check_call(["bash", "scripts/get_data.sh", "/content/data"])
        os.environ["SW_ROOT"] = "/content/data/spectralwaste_segmentation"

## Setup

Imports, the class palette, the output folder, and the nominal wavelength axis. `meta.json` ships no wavelengths, so the nm axis assumes even spacing of the 224 bands over 900 to 1700 nm and says so.

In [ ]:
import os
import sys
from pathlib import Path


def _find_repo_root():
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "src" / "sw" / "__init__.py").is_file():
            return cand
    raise RuntimeError("Start Jupyter from the repo root or from the notebooks/ folder.")


REPO = _find_repo_root()
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyBboxPatch, Patch

from sw.data import CLASS_NAMES, HYPER_LT_TRAIN_PIXEL_COUNTS, PALETTE

FIG_DIR = REPO / "docs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
SW_ROOT = os.environ.get("SW_ROOT")

OBJECT_CLASSES = CLASS_NAMES[1:]  # film .. bag, ids 1..6
COLOR = {name: np.array(rgb) / 255 for name, rgb in zip(CLASS_NAMES, PALETTE)}
LABEL = {**{name: name for name in CLASS_NAMES}, "video_tape": "tape"}
INK, MUTED = "#222222", "#6b6b6b"

N_BANDS, GROUP, N_GROUPS = 224, 16, 14
FALSE_COLOUR_BANDS = (56, 112, 168)  # notebook 01: C // 4, C // 2, 3C // 4


def nominal_nm(band):
    """Nominal wavelength. meta.json ships none; assume even spacing over 900-1700 nm."""
    return 900 + np.asarray(band, dtype=float) * 800 / (N_BANDS - 1)


def band_of_nm(nm):
    return (np.asarray(nm, dtype=float) - 900) * (N_BANDS - 1) / 800


plt.rcParams.update(
    {
        "font.size": 13,
        "axes.titlesize": 14,
        "axes.labelsize": 13,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "hatch.linewidth": 0.6,
        "savefig.dpi": 200,
        "savefig.bbox": "tight",
        "figure.facecolor": "white",
    }
)


def save(fig, name, svg=False):
    """Write docs/figures/<name>.png (and .svg when asked) and print the path."""
    png = FIG_DIR / f"{name}.png"
    fig.savefig(png)
    if svg:
        fig.savefig(FIG_DIR / f"{name}.svg")
    print("saved", png.relative_to(REPO))
    return png


print("repo", REPO)
print("SW_ROOT", SW_ROOT or "(not set: cells C and D will skip)")

## A. Per-class IoU bars (slide 7)

Ours is val, half resolution for cube and mask, 30 epochs, best epoch 29, one run. The reference ticks are the paper's MiniNet-v2 HYPER row, the fair comparison because MiniNetSeg is from the same family. Background (86.8) stays off: it is not in the mean.

In [ ]:
# docs/RESULTS.md, row cnn_ds2, best val FG mIoU 0.391 at epoch 29, in percent
OURS = {"film": 55.2, "basket": 49.5, "cardboard": 57.8, "video_tape": 28.6, "filament": 0.0, "bag": 43.6}
OURS_MEAN = 39.1
OURS_CONDITION = (
    "MiniNetSeg, 571,879 params; half resolution (cube and mask); 30 epochs; "
    "val split; best epoch 29; one run. Background (86.8) is not in the mean."
)
# Casao et al., IROS 2024, Table IV, MiniNet-v2 HYPER row, in percent
PAPER = {"film": 61.2, "basket": 61.0, "cardboard": 78.8, "video_tape": 28.8, "filament": 30.5, "bag": 56.3}
PAPER_MEAN = 52.8
PAPER_LABEL = "paper, MiniNet-v2 HYPER: full res, 200 epochs"

assert abs(np.mean(list(OURS.values())) - OURS_MEAN) < 0.05
assert abs(np.mean(list(PAPER.values())) - PAPER_MEAN) < 0.05

order = sorted(OBJECT_CLASSES, key=lambda c: OURS[c], reverse=True)  # filament last
y = np.arange(len(order))[::-1]  # first of `order` on top

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.barh(
    y,
    [OURS[c] for c in order],
    height=0.62,
    color=[COLOR[c] for c in order],
    edgecolor=INK,
    linewidth=0.8,
    label="ours, val, half resolution",
    zorder=2,
)
ax.scatter(
    [PAPER[c] for c in order], y, marker="|", s=520, color=INK, linewidths=2.4, label=PAPER_LABEL, zorder=3
)
for yi, c in zip(y, order):
    ax.text(OURS[c] + 1.0, yi, f"{OURS[c]:.1f}", va="center", fontsize=11, color=INK, zorder=4)
ax.axvline(OURS_MEAN, color=INK, linestyle="--", linewidth=1.2, zorder=1)
ax.axvline(PAPER_MEAN, color=MUTED, linestyle=":", linewidth=1.2, zorder=1)
ax.text(OURS_MEAN, y[0] + 0.62, f"FG mIoU {OURS_MEAN}", ha="center", va="bottom", fontsize=11, color=INK)
ax.text(PAPER_MEAN, y[0] + 0.62, f"paper {PAPER_MEAN}", ha="center", va="bottom", fontsize=11, color=MUTED)
ax.set_ylim(-0.6, y[0] + 1.25)
ax.set_yticks(y, [LABEL[c] for c in order])
ax.set_xlim(0, 100)
ax.set_xlabel("IoU (%)")
ax.grid(axis="x", color="#e9e9e9", zorder=0)
ax.legend(loc="lower right", frameon=False)
ax.set_title("Per-class IoU, first CNN run", loc="left")
fig.text(0.01, -0.04, OURS_CONDITION, fontsize=10, color=MUTED)
save(fig, "per_class_iou")
plt.show()

gap = {c: PAPER[c] - OURS[c] for c in OBJECT_CLASSES}
print("gap vs paper (points):", {LABEL[c]: round(v, 1) for c, v in gap.items()})
print("filament + cardboard share of the gap:", round((gap["filament"] + gap["cardboard"]) / sum(gap.values()), 2))

## B. Architecture strip, current vs goal (slide 8)

Shapes per stage at a 256 input, from `src/sw/models/transformer.py` on `main`. Status is box style plus a word, never colour alone: solid is merged, hatched is in review, dashed outline is a stub. The editable version for the slide is drawn in Slides from this same table; this cell is the fallback and writes an SVG too.

In [ ]:
STAGES = [
    # title, detail, output shape, params, status
    ("Input cube", "loader: uint16 scaled to [0, 1]", "(B, 224, 256, 256)", "0", "done"),
    (
        "Band tokenizer",
        "14 groups x 16 bands\nshared Conv2d 16 to 64, k8 s8\nLayerNorm + group emb. 14 x 64",
        "(B, 1024, 14, 64)",
        "66,624",
        "merged, PR #9",
    ),
    ("Spectral block x2", "attention over the 14 band\ngroups, inside each patch", "(B, 1024, 14, 64)", "49,984 each", "merged, PR #12"),
    ("Spectral pool", "mean over the 14 groups", "(B, 1024, 64)", "0", "merged, PR #12"),
    ("Spatial PE", "fixed 2D sin-cos\non the 32 x 32 grid", "(B, 1024, 64)", "0", "stub"),
    ("Spatial block x6", "attention over the\n1,024 patches", "(B, 1024, 64)", "49,984 each", "stub"),
    ("Seg head", "(B, 64, 32, 32) to 7 classes\nupsample x8", "(B, 7, 256, 256)", "~0.5k (1x1) to ~21k", "stub"),
    ("Loss + metric", "CE with paper-scheme weights\non the HSI counts;\nFG mIoU over classes 1 to 6", "scalar", "0", "done (harness)"),
]
PER_BLOCK, N_BLOCKS, TOKENIZER = 49_984, 8, 66_624
assert TOKENIZER + N_BLOCKS * PER_BLOCK == 466_496


def kind_of(status):
    if status.startswith("stub"):
        return "stub"
    if status.startswith("in review"):
        return "review"
    return "done"


def draw_box(x, y, w, h, status, rounding):
    """Solid = merged or done, hatched = in review, dashed outline = stub. Style plus a word, never colour alone."""
    kind = kind_of(status)
    style = f"round,pad=0,rounding_size={rounding}"
    if kind == "review":
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=style, facecolor="white", edgecolor="#a6a6a6", hatch="///", linewidth=0))
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=style, facecolor="none", edgecolor=INK, linewidth=1.4))
    elif kind == "stub":
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=style, facecolor="white", edgecolor=INK, linewidth=1.4, linestyle=(0, (4, 3))))
    else:
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=style, facecolor="#e6e6e6", edgecolor=INK, linewidth=1.4))


def text_bg(status):
    """A pad behind box text so hatching never runs through the words."""
    return dict(facecolor="#e6e6e6" if kind_of(status) == "done" else "white", edgecolor="none", pad=1.5)


W, H = 0.86, 0.66
ROW_Y = {0: 1.20, 1: 0.30}  # bottom of each row
fig, ax = plt.subplots(figsize=(13.33, 7.5))
ax.set_xlim(0, 4)
ax.set_ylim(0, 2.05)
ax.axis("off")
ax.text(0.03, 2.0, "Band-token transformer, spectral then spatial: current vs goal", fontsize=15, fontweight="bold", va="top", color=INK)

for i, (title, detail, shape, params, status) in enumerate(STAGES):
    row, col = divmod(i, 4)
    x0, y0 = 0.07 + col, ROW_Y[row]
    draw_box(x0, y0, W, H, status, 0.03)
    cx, bg = x0 + W / 2, text_bg(status)
    ax.text(cx, y0 + H - 0.05, title, ha="center", va="top", fontsize=12, fontweight="bold", color=INK, bbox=bg)
    ax.text(cx, y0 + H - 0.16, detail, ha="center", va="top", fontsize=9, color=INK, linespacing=1.25, bbox=bg)
    ax.text(cx, y0 + 0.19, shape, ha="center", va="center", fontsize=9.5, family="monospace", color=INK, bbox=bg)
    ax.text(cx, y0 + 0.11, f"params {params}", ha="center", va="center", fontsize=9, color=MUTED, bbox=bg)
    ax.text(cx, y0 + 0.04, status, ha="center", va="center", fontsize=9.5, style="italic", color=INK, bbox=bg)
    if col < 3:  # arrow to the next box in the row, below the detail text
        yc = y0 + 0.29
        ax.annotate("", xy=(x0 + 1.0, yc), xytext=(x0 + W, yc), arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.3))

# connector from the end of row 1 to the start of row 2
top_end = (0.07 + 3 + W / 2, ROW_Y[0])
mid_y = ROW_Y[0] - 0.12
start2 = (0.07 + W / 2, ROW_Y[1] + H)
ax.plot([top_end[0], top_end[0], start2[0]], [top_end[1], mid_y, mid_y], color=INK, lw=1.3)
ax.annotate("", xy=start2, xytext=(start2[0], mid_y), arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.3))

# legend of box styles, then the total and the footnote on their own lines
present = {kind_of(stage[4]) for stage in STAGES}
legend = [(w, s) for w, s in [("merged / done", "done"), ("in review", "in review"), ("stub, to do", "stub")] if kind_of(s) in present]
for k, (word, status) in enumerate(legend):
    draw_box(0.07 + 0.72 * k, 0.19, 0.14, 0.07, status, 0.015)
    ax.text(0.07 + 0.72 * k + 0.17, 0.225, word, va="center", fontsize=10, color=INK)
ax.text(
    0.07,
    0.12,
    f"Total, designed and not yet measured: {TOKENIZER:,} + {N_BLOCKS} x {PER_BLOCK:,} = {TOKENIZER + N_BLOCKS * PER_BLOCK:,} + head, about 0.47M (1x1 head) to 0.49M (recipe head).",
    va="center",
    fontsize=10,
    color=INK,
)
ax.text(
    0.07,
    0.04,
    "--downsample 2 (first Colab run): 16 x 16 grid, 256 tokens.   Goal: Phase 2 ablation (spectral only / spatial only / both); Phase 3 INT8, planned.",
    va="center",
    fontsize=10,
    color=MUTED,
)
save(fig, "architecture", svg=True)
plt.show()

## C. One cube: false colour, mask, mean spectrum per class (slide 6, or slide 3)

Needs `SW_ROOT`. Picks the train cube by scanning the masks only (no cube reads): filament present first, then the most classes. The stem is printed so the slide can cite it. The false-colour panel is three infrared bands, not what an RGB camera sees.

In [ ]:
if not SW_ROOT:
    print("SW_ROOT is not set; skipping cell C. See notebook 01 for the download, then re-run.")
else:
    from PIL import Image

    from sw.data import SpectralWasteDataset

    train = SpectralWasteDataset(split="train")
    best = None
    for index, (_cube_path, mask_path) in enumerate(train.pairs):
        with Image.open(mask_path) as img:
            ids = np.unique(np.array(img))
        key = (5 in ids, len(ids), -index)  # filament first, then the most classes, then the earliest stem
        if best is None or key > best[0]:
            best = (key, index)
    index = best[1]
    cube, mask = train[index]
    stem = train.pairs[index][0].stem
    cube_np, mask_np = cube.numpy(), mask.numpy()
    present = [c for c in range(len(CLASS_NAMES)) if (mask_np == c).any()]
    print(f"train[{index}] stem {stem}: {[CLASS_NAMES[c] for c in present]}")

    rgb = cube_np[list(FALSE_COLOUR_BANDS)].transpose(1, 2, 0).copy()
    for k in range(3):
        lo, hi = np.percentile(rgb[..., k], [2, 98])
        rgb[..., k] = np.clip((rgb[..., k] - lo) / (hi - lo + 1e-6), 0, 1)
    colour = np.zeros((*mask_np.shape, 3))
    for c, name in enumerate(CLASS_NAMES):
        colour[mask_np == c] = COLOR[name]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), gridspec_kw={"width_ratios": [1, 1, 1.6]})
    nm = ", ".join(f"{v:.0f}" for v in nominal_nm(FALSE_COLOUR_BANDS))
    axes[0].imshow(rgb)
    axes[0].set_title(f"false colour, bands {FALSE_COLOUR_BANDS}\napprox. {nm} nm (all infrared)", fontsize=11)
    axes[0].axis("off")
    axes[1].imshow(colour)
    axes[1].set_title("label mask (labels_hyper_lt)", fontsize=11)
    axes[1].axis("off")
    axes[1].legend(
        handles=[Patch(facecolor=COLOR[n], edgecolor="0.4", label=f"{i} {LABEL[n]}") for i, n in enumerate(CLASS_NAMES)],
        loc="upper center",
        bbox_to_anchor=(0.5, -0.01),
        ncol=4,
        frameon=False,
        fontsize=9,
    )
    bands = np.arange(N_BANDS)
    for c in present:
        name = CLASS_NAMES[c]
        spectrum = cube_np[:, mask_np == c].mean(axis=1)
        if name == "background":
            axes[2].plot(bands, spectrum, color="0.55", linewidth=1.2, linestyle=":", label="background")
        else:
            axes[2].plot(bands, spectrum, color=COLOR[name], linewidth=2, label=LABEL[name])
        axes[2].text(bands[-1] + 3, spectrum[-1], LABEL[name], va="center", fontsize=10, color=INK)
    axes[2].set_xlim(0, N_BANDS + 20)
    axes[2].set_ylim(0, 1)
    axes[2].set_xlabel("band index")
    axes[2].set_ylabel("scaled value")
    axes[2].secondary_xaxis("top", functions=(nominal_nm, band_of_nm)).set_xlabel("nominal nm (even spacing assumed)", fontsize=10)
    axes[2].grid(color="#eeeeee")
    axes[2].legend(frameon=False, fontsize=10, loc="upper left")
    axes[2].set_title("mean spectrum per class, this image", fontsize=11)
    fig.suptitle(f"One train cube ({stem}): 224 x 256 x 256, uint16 scaled to [0, 1]", x=0.01, ha="left", fontsize=13)
    save(fig, "data_example")
    plt.show()

## D. Mean spectrum per class over N train cubes (slide 3 or 6)

Needs `SW_ROOT`. Every class gets a curve, with a plus or minus one standard deviation band, the 14 token groups shaded, and the three false-colour bands marked. `N_CUBES = 50` sorted stems reads about 1.5 GB; `len(train)` reads the whole split, about 15 GB, roughly ten minutes from the Colab local disk. The right panel is the class pixel share from the train counts in `sw.data`.

In [ ]:
N_CUBES = 50  # sorted stems; len(train) for the whole split

if not SW_ROOT:
    print("SW_ROOT is not set; skipping cell D.")
else:
    from sw.data import SpectralWasteDataset

    train = SpectralWasteDataset(split="train")
    n = min(N_CUBES, len(train))
    K = len(CLASS_NAMES)
    total = np.zeros((K, N_BANDS))
    total_sq = np.zeros((K, N_BANDS))
    count = np.zeros(K)
    for index in range(n):
        cube, mask = train[index]
        cube_np, mask_np = cube.numpy().astype(np.float64), mask.numpy()
        for c in np.unique(mask_np):
            pix = cube_np[:, mask_np == c]  # (224, n_pixels)
            total[c] += pix.sum(axis=1)
            total_sq[c] += (pix**2).sum(axis=1)
            count[c] += pix.shape[1]
    denom = np.maximum(count, 1)[:, None]
    mean = total / denom
    std = np.sqrt(np.maximum(total_sq / denom - mean**2, 0))
    print("pixels per class in these cubes:", {LABEL[n_]: int(c_) for n_, c_ in zip(CLASS_NAMES, count)})

    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [3, 1]})
    bands = np.arange(N_BANDS)
    for g in range(N_GROUPS):
        if g % 2 == 0:
            ax.axvspan(g * GROUP - 0.5, g * GROUP + GROUP - 0.5, color="black", alpha=0.045, linewidth=0)
    for b in FALSE_COLOUR_BANDS:
        ax.axvline(b, color=MUTED, linewidth=0.8, linestyle=":")
    for c, name in enumerate(CLASS_NAMES):
        if count[c] == 0:
            continue
        if name == "background":
            ax.plot(bands, mean[c], color="0.55", linewidth=1.2, linestyle=":", label="background")
            continue
        ax.plot(bands, mean[c], color=COLOR[name], linewidth=2, label=LABEL[name])
        ax.fill_between(bands, mean[c] - std[c], mean[c] + std[c], color=COLOR[name], alpha=0.12, linewidth=0)
        ax.text(bands[-1] + 3, mean[c, -1], LABEL[name], va="center", fontsize=10, color=INK)
    ax.set_xlim(0, N_BANDS + 20)
    ax.set_ylim(0, 1)
    ax.set_xlabel("band index   (shaded: the 14 token groups of 16 bands; dotted: the false-colour bands)")
    ax.set_ylabel("mean scaled value, band = 1 sd")
    ax.secondary_xaxis("top", functions=(nominal_nm, band_of_nm)).set_xlabel(
        "nominal nm (even spacing assumed; meta.json ships no wavelengths)", fontsize=10
    )
    ax.legend(frameon=False, ncol=2, loc="upper left", fontsize=10)
    ax.set_title(f"Mean spectrum per class over {n} train cubes", loc="left")

    share = np.array(HYPER_LT_TRAIN_PIXEL_COUNTS) / sum(HYPER_LT_TRAIN_PIXEL_COUNTS) * 100
    ys = np.arange(K)[::-1]
    ax2.barh(ys, share, color=[COLOR[n_] for n_ in CLASS_NAMES], edgecolor=INK, linewidth=0.8)
    ax2.set_yticks(ys, [LABEL[n_] for n_ in CLASS_NAMES])
    ax2.set_xscale("log")
    ax2.set_xlim(0.03, 300)
    ax2.set_xlabel("% of train pixels (log)")
    for yi, s in zip(ys, share):
        ax2.text(s * 1.2, yi, f"{s:.3g}%", va="center", fontsize=10, color=INK)
    ax2.set_title("Class pixel share, whole train split", loc="left", fontsize=12)
    save(fig, "band_signatures")
    plt.show()

## G. Training curves from `metrics.csv` (Nov 5)

The harness writes `epoch, train_loss, val_fg_miou, steps, iou_<class>` per epoch. Copy the CNN run's file from Drive to `docs/metrics/cnn_ds2.csv` (a few KB; `docs/runs/` would be caught by the `runs/` ignore rule) and point the transformer entry at its run folder. Entries whose file is missing are skipped.

In [ ]:
import csv

RUNS = {
    "CNN control, cnn_ds2": REPO / "docs" / "metrics" / "cnn_ds2.csv",
    "band-token transformer": REPO / "runs" / "transformer" / "metrics.csv",
}
found = {name: path for name, path in RUNS.items() if path.is_file()}
if not found:
    print("No metrics file found. Copy cnn_ds2/metrics.csv to docs/metrics/cnn_ds2.csv, or edit RUNS.")
else:
    fig, (ax_l, ax_m) = plt.subplots(1, 2, figsize=(12, 4.5))
    for (name, path), ls in zip(found.items(), ["-", "--", ":"]):
        with path.open(newline="") as handle:
            rows = list(csv.DictReader(handle))
        epochs = np.array([int(r["epoch"]) for r in rows])
        loss = np.array([float(r["train_loss"]) for r in rows])
        miou = np.array([float(r["val_fg_miou"]) * 100 if r["val_fg_miou"] else np.nan for r in rows])
        ax_l.plot(epochs, loss, ls, color=INK, linewidth=2, label=name)
        ax_m.plot(epochs, miou, ls, color=INK, linewidth=2, label=name)
        best = int(np.nanargmax(miou))
        ax_m.scatter(epochs[best], miou[best], s=50, color=INK, zorder=3)
        ax_m.text(epochs[best], miou[best] + 1.5, f"{miou[best]:.1f} (epoch {epochs[best]})", ha="center", fontsize=10, color=INK)
    ax_l.set_xlabel("epoch")
    ax_l.set_ylabel("train loss")
    ax_l.set_title("Train loss", loc="left")
    ax_l.grid(color="#eeeeee")
    ax_m.set_xlabel("epoch")
    ax_m.set_ylabel("val FG mIoU (%)")
    ax_m.set_title("Val FG mIoU, half resolution", loc="left")
    ax_m.grid(color="#eeeeee")
    ax_m.set_ylim(0, 60)
    ax_m.axhline(52.8, color=MUTED, linestyle=":", linewidth=1)
    ax_m.text(1, 53.6, "paper MiniNet-v2 52.8 (full res, 200 epochs)", fontsize=9, color=MUTED)
    ax_m.legend(frameon=False, loc="lower right")
    save(fig, "training_curves")
    plt.show()